In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pickle
import glob

In [5]:
df = pd.DataFrame(columns=['model', 'dataset', 'method', 'args', 'iter', 'acc@1', 'acc@5'])

paths = glob.glob('./output/*/*/*')
for path in sorted(paths):
    if not os.path.isdir(path):
        continue

    model, dataset, method = path.split('/')[-3:]
    args = ', '.join(method.split('_')[1:])
    method = method.split('_')[0]
    acc1 = []
    acc5 = []

    for iter in os.listdir(path):
        if not os.path.exists(os.path.join(path, iter, 'results.pkl')):
            continue
        
        results = pickle.load(open(os.path.join(path, iter, 'results.pkl'), 'rb'))

        acc1.append(results['accuracy1'])
        acc5.append(results['accuracy5'])
        
    if len(acc1) > 0:
        df = df.append({
            'model': model,
            'dataset': dataset,
            'method': method,
            'args': args,
            'iter': len(acc1),
            'acc@1': f'{np.mean(acc1):.2f}±{np.std(acc1):.2f}',
            'acc@5': f'{np.mean(acc5):.2f}±{np.std(acc5):.2f}'
        }, ignore_index=True)
        # print(path)
print(df)


        model  dataset    method args iter       acc@1       acc@5
0  eswin-tiny  food101  baseline         1  86.85±0.00  96.90±0.00


In [ ]:
from models.eswin_transformer import ESwinTransformer

import torch.nn as nn
import torch


model = ESwinTransformer(window_size=4, mlp_ratio=2, img_size=128, patch_size=2)
model.load_state_dict(torch.load('./output/eswin-tiny/food101/baseline/0/ckpt_best.pt'), strict=False)

plt.figure(figsize=(10, 6))
idx = 0
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        weight = module.weight.data
        singular_values = torch.linalg.svd(weight).S
        plt.scatter([idx]*len(singular_values), singular_values.cpu().numpy(), label=name if idx == 0 else "", alpha=0.6, c='C0')
        color = 'red' if 'qkv' in name else 'blue' if 'proj' in name else 'green' if 'mlp' in name else 'black'
        plt.scatter(idx, singular_values.max().cpu().numpy(), color=color, marker='x')
        idx += 1

In [52]:
import math
import weightwatcher as ww
import torch
from torchvision import models

model_name = 'resnet34'
datasets = ['cifar10', 'cifar100', 'tiny-imagenet', 'stl10', 'food101']
for dataset in datasets:
    dataset_path = f'./output/{model_name}/{dataset}'
    if not os.path.exists(dataset_path):
        continue
    for path in sorted(os.listdir(dataset_path)):
        if not os.path.exists(os.path.join(dataset_path, path, '0', 'ckpt_best.pt')):
            continue
        num_classes = 10 if dataset in ['cifar10', 'stl10'] else 100 if dataset == 'cifar100' else 200 if dataset == 'tiny-imagenet' else 101
        model = models.resnet34(num_classes=num_classes)
        model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        model.maxpool = torch.nn.Identity()

        ckpt = torch.load(os.path.join(dataset_path, path, '0', 'ckpt_best.pt'))
        model.load_state_dict(ckpt['model'], strict=False)

        watcher = ww.WeightWatcher(model=model)
        details = watcher.analyze(plot=False, layers=[])
        
        print(dataset, path, f'{details["alpha"].mean():.2f}')
    print()


AttributeError: module 'torch.linalg' has no attribute 'svdvals'